# Dice Throne: hero vs hero shaping
Load, dedupe, and reshape games into hero-vs-hero results. No modelling yet.

In [1]:
import json
from pathlib import Path

import pandas as pd

DATA_FILE = Path('dt-data-for-log-reg.txt')
MIN_GAMES = 5  # threshold for a hero pair to count as 'enough' data

## 1. Import
The file holds two JSON arrays back to back, so decode them one at a time.

In [2]:
text = DATA_FILE.read_text(encoding='utf-8')
decoder = json.JSONDecoder()
arrays, pos = [], 0
while True:
    while pos < len(text) and text[pos].isspace():
        pos += 1
    if pos >= len(text):
        break
    arr, pos = decoder.raw_decode(text, pos)
    arrays.append(arr)

print('records per array:', [len(a) for a in arrays])
raw = pd.DataFrame([r for a in arrays for r in a])
print('total raw records:', len(raw))
raw.head(3)

records per array: [19, 14]
total raw records: 33


,ts,app,gsi1sk,user,pk,game,gsi1pk,sk
0,2026-09-25T16:14:31.965Z,tca-dice-throne-26s,2026-09-25T16:14:31.965Z,Joshua105@yahoo.com,joshua105@yahoo.com,"{'start': '2026-09-25T16:14:28.723Z', 'winner'...",tca-dice-throne-26s,tca-dice-throne-26s#2026-09-25T16:14:31.965Z
1,2026-09-25T16:14:44.167Z,tca-dice-throne-26s,2026-09-25T16:14:44.167Z,Joshua105@yahoo.com,joshua105@yahoo.com,"{'start': '2026-09-25T16:14:42.179Z', 'end': '...",tca-dice-throne-26s,tca-dice-throne-26s#2026-09-25T16:14:44.167Z
2,2026-09-25T16:59:29.228Z,tca-dice-throne-26s,2026-09-25T16:59:29.228Z,Joshua105@yahoo.com,joshua105@yahoo.com,"{'end': '2026-09-25T16:59:29.228Z', 'players':...",tca-dice-throne-26s,tca-dice-throne-26s#2026-09-25T16:59:29.228Z


## 2. Dedupe
A game is identified by start + end (not user, since games were copied between users).

In [3]:
raw['game_start'] = raw['game'].map(lambda g: g.get('start'))
raw['game_end'] = raw['game'].map(lambda g: g.get('end'))
key = ['game_start', 'game_end']  # games were copied between users, so user is not part of the key
print('duplicate records:', raw.duplicated(key).sum())
# do any duplicates disagree on content?
raw['game_json'] = raw['game'].map(lambda g: json.dumps(g, sort_keys=True))
print('keys with conflicting game content:', (raw.groupby(key)['game_json'].nunique() > 1).sum())
dedup = raw.drop_duplicates(key).reset_index(drop=True)
print('after dedupe:', len(dedup))
raw.groupby(key)['user'].nunique().value_counts().rename('games by number of users holding them')

duplicate records: 13
keys with conflicting game content: 0
after dedupe: 20


user
2    13
1     7
Name: games by number of users holding them, dtype: int64

## 3. Flatten to one row per game

In [4]:
rows, problems = [], []
for _, r in dedup.iterrows():
    g = r['game']
    players = g.get('players', [])
    winner = g.get('winner')
    if len(players) != 2:
        problems.append((r['game_start'], 'players != 2'))
        continue
    if not winner:
        problems.append((r['game_start'], 'no winner'))
        continue
    names = [p['name'] for p in players]
    if winner not in names:
        problems.append((r['game_start'], 'winner not a player'))
        continue
    p1, p2 = players
    rows.append({'user': r['user'], 'start': g['start'], 'end': g.get('end'),
                 'p1_name': p1['name'], 'p1_hero': p1.get('hero'),
                 'p2_name': p2['name'], 'p2_hero': p2.get('hero'),
                 'winner_name': winner})
games = pd.DataFrame(rows)
games['winner_hero'] = games.apply(lambda x: x.p1_hero if x.winner_name == x.p1_name else x.p2_hero, axis=1)
print('usable games:', len(games), '| dropped:', pd.Series([p[1] for p in problems]).value_counts().to_dict())
print('missing hero:', games[['p1_hero', 'p2_hero']].isna().sum().to_dict())
games.head()

usable games: 20 | dropped: {}
missing hero: {'p1_hero': 0, 'p2_hero': 0}


,user,start,end,p1_name,p1_hero,p2_name,p2_hero,winner_name,winner_hero
0,Joshua105@yahoo.com,2026-09-25T16:14:28.723Z,2026-09-25T16:14:31.965Z,Josh,Headless Horseman,Tom,Pale Lady,Tom,Pale Lady
1,Joshua105@yahoo.com,2026-09-25T16:14:42.179Z,2026-09-25T16:14:44.167Z,Josh,Headless Horseman,Tom,Pale Lady,Josh,Headless Horseman
2,Joshua105@yahoo.com,2026-09-25T16:59:27.516Z,2026-09-25T16:59:29.228Z,Josh,Scarlet Witch,Tom,Spiderman,Tom,Spiderman
3,Joshua105@yahoo.com,2026-09-25T16:59:44.150Z,2026-09-25T16:59:45.714Z,Josh,Loki,Tom,Spiderman,Tom,Spiderman
4,Joshua105@yahoo.com,2026-09-25T16:59:53.445Z,2026-09-25T16:59:55.069Z,Josh,Loki,Tom,Spiderman,Tom,Spiderman


In [5]:
# which hero does each player use?
pl = pd.concat([games[['p1_name', 'p1_hero']].set_axis(['name', 'hero'], axis=1),
                games[['p2_name', 'p2_hero']].set_axis(['name', 'hero'], axis=1)])
pl.groupby(['name', 'hero']).size().unstack(fill_value=0)

hero,Gunslinger,Headless Horseman,Loki,Moon Elf,Ninja,Pale Lady,Samurai,Scarlet Witch,Spiderman,Sun Elf,Thor,Treant
name,,,,,,,,,,,,
Anna,0,1,1,1,0,2,0,0,0,1,0,0
Josh,1,4,5,2,0,0,2,2,1,0,1,1
Taylor,0,0,0,0,1,0,0,0,0,0,0,0
Tom,0,0,0,0,1,4,2,0,5,1,1,0


## 4. Hero vs hero rows
One row per game per orientation (hero_a perspective), so each game appears twice with `a_won` flipped.

In [6]:
a = pd.DataFrame({'start': games.start, 'hero_a': games.p1_hero, 'hero_b': games.p2_hero, 'a_won': (games.winner_name == games.p1_name).astype(int)})
b = pd.DataFrame({'start': games.start, 'hero_a': games.p2_hero, 'hero_b': games.p1_hero, 'a_won': (games.winner_name == games.p2_name).astype(int)})
h2h = pd.concat([a, b], ignore_index=True)
print(len(h2h), 'rows')
h2h.head()

40 rows


,start,hero_a,hero_b,a_won
0,2026-09-25T16:14:28.723Z,Headless Horseman,Pale Lady,0
1,2026-09-25T16:14:42.179Z,Headless Horseman,Pale Lady,1
2,2026-09-25T16:59:27.516Z,Scarlet Witch,Spiderman,0
3,2026-09-25T16:59:44.150Z,Loki,Spiderman,0
4,2026-09-25T16:59:53.445Z,Loki,Spiderman,0


## 5. Which matchups have enough data?

In [7]:
hero_totals = h2h.groupby('hero_a').agg(games=('a_won', 'size'), wins=('a_won', 'sum'))
hero_totals['win_rate'] = (hero_totals.wins / hero_totals.games).round(2)
hero_totals.sort_values('games', ascending=False)

,games,wins,win_rate
hero_a,,,
Loki,6,2,0.33
Pale Lady,6,2,0.33
Spiderman,6,3,0.50
Headless Horseman,5,3,0.60
Samurai,4,2,0.50
Moon Elf,3,3,1.00
Scarlet Witch,2,1,0.50
Ninja,2,0,0.00
Thor,2,2,1.00


In [8]:
pairs = h2h.groupby(['hero_a', 'hero_b']).agg(games=('a_won', 'size'), a_wins=('a_won', 'sum')).reset_index()
pairs['a_win_rate'] = (pairs.a_wins / pairs.games).round(2)
# unordered view: each pair once
unordered = pairs[pairs.hero_a < pairs.hero_b].sort_values('games', ascending=False)
unordered[unordered.games >= MIN_GAMES] if (unordered.games >= MIN_GAMES).any() else unordered.head(15)

,hero_a,hero_b,games,a_wins,a_win_rate
8,Loki,Spiderman,3,1,0.33
2,Headless Horseman,Pale Lady,2,1,0.50
10,Moon Elf,Sun Elf,2,2,1.00
3,Headless Horseman,Samurai,1,1,1.00
4,Headless Horseman,Spiderman,1,1,1.00
5,Loki,Pale Lady,1,1,1.00
6,Loki,Samurai,1,0,0.00
0,Gunslinger,Headless Horseman,1,1,1.00
7,Loki,Scarlet Witch,1,0,0.00
9,Moon Elf,Samurai,1,1,1.00


In [9]:
# games-per-matchup matrix (symmetric)
counts = pairs.pivot(index='hero_a', columns='hero_b', values='games').fillna(0).astype(int)
counts.style.background_gradient(cmap='Blues')

hero_b,Gunslinger,Headless Horseman,Loki,Moon Elf,Ninja,Pale Lady,Samurai,Scarlet Witch,Spiderman,Sun Elf,Thor,Treant
hero_a,,,,,,,,,,,,
Gunslinger,0,1,0,0,0,0,0,0,0,0,0,0
Headless Horseman,1,0,0,0,0,2,1,0,1,0,0,0
Loki,0,0,0,0,0,1,1,1,3,0,0,0
Moon Elf,0,0,0,0,0,0,1,0,0,2,0,0
Ninja,0,0,0,0,0,1,0,0,0,0,1,0
Pale Lady,0,2,1,0,1,0,1,0,0,0,0,1
Samurai,0,1,1,1,0,1,0,0,0,0,0,0
Scarlet Witch,0,0,1,0,0,0,0,0,1,0,0,0
Spiderman,0,1,3,0,0,0,0,1,0,0,1,0


In [10]:
# win rate of row hero vs column hero
rates = pairs.pivot(index='hero_a', columns='hero_b', values='a_win_rate')
rates.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1).format('{:.2f}', na_rep='')

hero_b,Gunslinger,Headless Horseman,Loki,Moon Elf,Ninja,Pale Lady,Samurai,Scarlet Witch,Spiderman,Sun Elf,Thor,Treant
hero_a,,,,,,,,,,,,
Gunslinger,,1.00,,,,,,,,,,
Headless Horseman,0.00,,,,,0.50,1.00,,1.00,,,
Loki,,,,,,1.00,0.00,0.00,0.33,,,
Moon Elf,,,,,,,1.00,,,1.00,,
Ninja,,,,,,0.00,,,,,0.00,
Pale Lady,,0.50,0.00,,1.00,,0.00,,,,,0.00
Samurai,,0.00,1.00,0.00,,1.00,,,,,,
Scarlet Witch,,,1.00,,,,,,0.00,,,
Spiderman,,0.00,0.67,,,,,1.00,,,0.00,
